# AccessAI demo
A short demo of AccessAI: a simple accessibility scanner and conservative fixer. The goal is to show how the pipeline works with local HTML pages.


# Setup
This notebook runs locally and on Kaggle without external API keys. It demonstrates the scanner, fixer, and patcher using the sample pages provided in the repo. Follow cells in order and run them top-to-bottom.


In [1]:
# Try importing BeautifulSoup, install requirements if missing (best-effort)
import sys, os
try:
    import bs4
    print('bs4 available')
except Exception:
    print('bs4 not found, installing requirements (best-effort)')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'capstone-project/requirements.txt'])
    import bs4
    print('installed requirements')


bs4 available


In [2]:
# Import project agents
from agents import scanner, fixer, patcher

# Lightweight code_exec helper (demo-only)
def code_exec(code: str):
    """Run a short Python snippet and return stdout+returncode.
    This is a demo helper used only for small verification snippets.
    """
    import subprocess, sys
    try:
        res = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True, timeout=5)
        return {'stdout': res.stdout, 'stderr': res.stderr, 'returncode': res.returncode}
    except Exception as e:
        return {'stdout': '', 'stderr': str(e), 'returncode': -1}

print('agents imported:', scanner.__name__, fixer.__name__, patcher.__name__)


agents imported: agents.scanner agents.fixer agents.patcher


## Demo helper: run scan, suggest fixes, apply patches
This helper reads an HTML file, runs the scanner, asks the fixer for conservative suggestions (with optional verification), and prints short, readable results. The goal is to make the demo easy to run in other Notebook editors and compilers.


In [3]:
from pathlib import Path
import html

def demo_file(path):
    path = Path(path)
    html_text = path.read_text(encoding='utf-8')
    print(f'--- Demo: {path.name} ---')
    base_scan = scanner.analyze_html(html_text, str(path))
    print('Baseline:', base_scan.get('summary'))
    for issue in base_scan.get('issues', [])[:10]:
        print('-', issue['id'], '-', issue['message'])

    suggestions = fixer.suggest_fixes(base_scan, html_text, tools={'scanner': scanner.analyze_html, 'code_exec': code_exec})
    if not suggestions:
        print('No automatic suggestions produced.')
        return base_scan

    for i, s in enumerate(suggestions, start=1):
        issue = s.get('issue', {})
        print(f'\nSuggestion {i}: {s.get("suggestion")}')
        print(' Issue:', issue.get('id'), '-', issue.get('message'))
        print(' Patch (human-readable):', s.get('patch'))
        if 'verification' in s:
            print(' Verification stdout:', s['verification'].get('stdout'))
        if 'post_scan' in s:
            post = s['post_scan']
            print(' Post-scan summary:', post.get('summary'))
            print(' Reduction:', s.get('reduction'))
            # show a small snippet of the patched HTML for context
            patched = patcher.apply_patch_to_html(html_text, issue, s.get('patch', ''))
            snippet = patched[:500].replace('\n',' ')
            print(' Patched snippet:', snippet[:400] + ('...' if len(snippet)>400 else ''))

    return suggestions


## Run the demo on included sample pages
The repository includes several sample pages in `capstone-project/data/sample_pages/`. The cells below run the demo helper on each page and print concise results.


In [4]:
from pathlib import Path
import os

def find_repo_root():
    p = Path.cwd()
    for _ in range(20):
        if (p / 'capstone-project').exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    # fallback to known workspace path
    candidate = Path('/workspaces/fraudshield-workforce')
    if candidate.exists():
        return candidate
    return Path.cwd()

repo_root = find_repo_root()
pages_dir = repo_root / 'capstone-project' / 'data' / 'sample_pages'
if not pages_dir.exists():
    print('Sample pages directory not found at', pages_dir)
    candidates = list(repo_root.glob('**/sample_pages'))
    if candidates:
        print('Found sample_pages candidates:')
        for c in candidates:
            print(' -', c)
    else:
        raise FileNotFoundError(f'Could not locate sample_pages under {repo_root}')

for p in sorted(pages_dir.iterdir()):
    print('\n')
    demo_file(p)




--- Demo: bad.html ---
Baseline: 3 issue(s) found
- alt_missing - Image missing alt text
- heading_structure - Document should start with an H1
- label_missing_id - Form control missing id so label cannot be associated and missing aria-label

Suggestion 1: Add alt attribute: alt="Describe image"
 Issue: alt_missing - Image missing alt text
 Patch (human-readable): add alt to img:nth-of-type(1)
 Verification stdout: fail
 Post-scan summary: 2 issue(s) found
 Reduction: 1
 Patched snippet: <!DOCTYPE html>  <html> <head> <meta charset="utf-8"/> <title>Bad sample</title> </head> <body> <h2>Subtitle without H1</h2> <p>Images without alt and unlabeled inputs.</p> <img alt="Describe image" src="/img/missing.png"/> <form> <input type="text"/> </form> </body> </html> 

Suggestion 2: Ensure the page has a single H1 at top
 Issue: heading_structure - Document should start with an H1
 Patch (human-readable): <h1>Page title</h1>
 Verification stdout: ok
 Post-scan summary: 2 issue(s) found
 Reduc

## Interactive patch preview
Use the widget below to review suggested fixes, toggle which suggestions to apply, and preview the patched HTML before saving.


In [5]:
import html
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pathlib import Path

# UI to inspect and preview suggested patches for a local HTML file

def show_suggestions_ui(path):
    repo_root = globals().get('repo_root') or Path.cwd()
    p = Path(path)
    if not p.exists():
        p = repo_root / path
    if not p.exists():
        print('File not found:', path)
        return
    html_text = p.read_text(encoding='utf-8')
    scan = scanner.analyze_html(html_text, str(p))
    suggestions = fixer.suggest_fixes(scan, html_text, tools={'scanner': scanner.analyze_html, 'code_exec': code_exec})
    if not suggestions:
        print('No suggestions for', p.name)
        return

    boxes = []
    for i, s in enumerate(suggestions):
        cb = widgets.Checkbox(value=True, description=f"{s.get('suggestion')} [{s.get('issue',{}).get('id')}]")
        boxes.append(cb)

    out = widgets.Output()

    def get_patched_html():
        patched = html_text
        for i, s in enumerate(suggestions):
            if boxes[i].value:
                patched = patcher.apply_patch_to_html(patched, s.get('issue', {}), s.get('patch', ''))
        return patched

    def on_preview(b):
        with out:
            clear_output()
            patched = get_patched_html()
            display(HTML('<h4>Patched HTML Preview</h4>'))
            display(HTML('<pre style="white-space: pre-wrap;max-height:600px;overflow:auto;">'+html.escape(patched)[:8000]+'</pre>'))

    def on_show_full(b):
        with out:
            clear_output()
            patched = get_patched_html()
            display(HTML('<h4>Patched HTML (full)</h4>'))
            display(HTML('<pre style="white-space: pre-wrap;">'+html.escape(patched)+'</pre>'))

    preview_btn = widgets.Button(description='Preview selected patches')
    full_btn = widgets.Button(description='Show patched HTML (full)')
    preview_btn.on_click(on_preview)
    full_btn.on_click(on_show_full)

    header = widgets.HTML(value=f"<b>Suggestions for {p.name}</b>")
    btns = widgets.HBox([preview_btn, full_btn])
    ui = widgets.VBox([header] + boxes + [btns, out])
    display(ui)

# Helper usage hint
print('UI helper loaded. Call `show_suggestions_ui("capstone-project/data/sample_pages/bad.html")` to try it.')


UI helper loaded. Call `show_suggestions_ui("capstone-project/data/sample_pages/bad.html")` to try it.


In [6]:
from pathlib import Path

# Create patched version for a sample and save to file, then print a short preview
p = Path('capstone-project/data/sample_pages/bad.html')
if not p.exists():
    p = find_repo_root() / 'capstone-project' / 'data' / 'sample_pages' / 'bad.html'
html_text = p.read_text(encoding='utf-8')
scan = scanner.analyze_html(html_text, str(p))
suggestions = fixer.suggest_fixes(scan, html_text, tools={'scanner': scanner.analyze_html, 'code_exec': code_exec})
patched = html_text
for s in suggestions:
    patched = patcher.apply_patch_to_html(patched, s.get('issue', {}), s.get('patch', ''))

out_path = Path('capstone-project/tmp/patched_bad.html')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(patched, encoding='utf-8')
print('Patched HTML written to', out_path)
preview = patched[:4000]
if len(patched) > 4000:
    preview += '\n\n... (truncated) ...\n'
print(preview)


Patched HTML written to capstone-project/tmp/patched_bad.html
<!DOCTYPE html>

<html>
<head>
<meta charset="utf-8"/>
<title>Bad sample</title>
</head>
<body><h1>Page title</h1>
<h2>Subtitle without H1</h2>
<p>Images without alt and unlabeled inputs.</p>
<img alt="Describe image" src="/img/missing.png"/>
<form>
<label>Label</label><input aria-label="Label" type="text"/>
</form>
</body>
</html>



In [7]:
# Generate patched output for `medium.html`, save file, and print suggestions + preview
from pathlib import Path
sample = 'medium.html'
repo = globals().get('repo_root', Path.cwd())
pp = repo / 'capstone-project' / 'data' / 'sample_pages' / sample
if not pp.exists():
    pp = Path('capstone-project/data/sample_pages') / sample
    if not pp.exists():
        raise FileNotFoundError(pp)

html_text = pp.read_text(encoding='utf-8')
scan = scanner.analyze_html(html_text, str(pp))
print('Scan summary:', scan.get('summary'))

suggestions = fixer.suggest_fixes(scan, html_text, tools={'scanner': scanner.analyze_html, 'code_exec': code_exec})
if not suggestions:
    print('No suggestions produced for', sample)
else:
    print(f"\nFound {len(suggestions)} suggestion(s):")
    for i, s in enumerate(suggestions, start=1):
        issue = s.get('issue', {})
        print(f"\nSuggestion {i} — Issue: {issue.get('id')} | Message: {issue.get('message')}")
        print(' Suggestion text:', s.get('suggestion'))
        print(' Patch snippet:', (s.get('patch') or '')[:300].replace('\n',' '))
        if 'post_scan' in s:
            print(' Post-scan summary:', s['post_scan'].get('summary'))
        if 'verification' in s:
            print(' Verification stdout:', s['verification'].get('stdout'))

# Apply all suggestions conservatively and save patched file
patched = html_text
for s in suggestions:
    patched = patcher.apply_patch_to_html(patched, s.get('issue', {}), s.get('patch', ''))

out_path = Path('capstone-project/tmp/patched_medium.html')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(patched, encoding='utf-8')
print('\nPatched HTML written to', out_path)

preview = patched[:3000]
if len(patched) > 3000:
    preview += '\n\n... (truncated) ...\n'
print('\n--- Patched preview (first 3000 chars) ---')
print(preview)


Scan summary: 3 issue(s) found

Found 3 suggestion(s):

Suggestion 1 — Issue: alt_missing | Message: Image missing alt text
 Suggestion text: Add alt attribute: alt="Describe image"
 Patch snippet: add alt to img:nth-of-type(2)
 Post-scan summary: 2 issue(s) found
 Verification stdout: fail

Suggestion 2 — Issue: heading_structure | Message: Document should start with an H1
 Suggestion text: Ensure the page has a single H1 at top
 Patch snippet: <h1>Page title</h1>
 Post-scan summary: 2 issue(s) found
 Verification stdout: ok

Suggestion 3 — Issue: label_missing | Message: Form control with id "email" missing label or aria-label
 Suggestion text: Add associated <label> or aria-label to form control
 Patch snippet: insert label for control
 Post-scan summary: 2 issue(s) found

Patched HTML written to capstone-project/tmp/patched_medium.html

--- Patched preview (first 3000 chars) ---
<!DOCTYPE html>

<html>
<head>
<meta charset="utf-8"/>
<title>Medium sample</title>
</head>
<body><h1>Pa